<a href="https://colab.research.google.com/github/ricardoescu/NumberEncoding/blob/Gemini/Number_embedding_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers datasets torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
!pip install word2number

  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=3c1a7ad0f33e1e66c4198aaa8b1429875257f66e2616bcd745cf695e015f1ac1
  Stored in directory: /root/.cache/pip/wheels/cd/ef/ae/073b491b14d25e2efafcffca9e16b2ee6d114ec5c643ba4f06
Successfully built word2number


In [ ]:

import os
os.environ["WANDB_DISABLED"] = "true"

import logging
import pandas as pd
from sentence_transformers import SentenceTransformer, losses, InputExample, LoggingHandler
from torch.utils.data import DataLoader
from collections import defaultdict
import numpy as np
import re
from word2number import w2n

def _parse_age(sentence: str) -> int:
    m = re.search(r"\b(\d+)\b", sentence)
    if m:
        return int(m.group(1))
    txt = sentence.lower().split("my age is")[-1]
    txt = re.sub(r"[^a-z\s-]", " ", txt).strip()
    return w2n.word_to_num(txt)


df    = pd.read_csv("cleaned_data.csv", header=0)
def row_to_sentence(row):
    return ", ".join(f"the {col} is {row[col]}" for col in df.columns)

lines = df.apply(row_to_sentence, axis=1).tolist()#lines = df.iloc[:,0].astype(str).tolist()
ages  = [_parse_age(s) for s in lines]
idx_by_age = list(enumerate(ages))

OBJECTIVE = "MNRL" #param ["triplet", "MNRL"] -> Multiple Negative Ranking Loss
MODEL_ID  = "bert-base-uncased"
EPOCHS    = 3
BATCH     = 32
OUT_DIR   = "bert_MNRL_ACS"


examples = []
if OBJECTIVE == "triplet":
    # triplet anchors: (anchor_sent, nearest_age_sent, farthest_age_sent)
    for i, age in idx_by_age:
        diffs = [(j, abs(age - other)) for j, other in idx_by_age if j != i]
        pos_i, _ = min(diffs, key=lambda x: x[1])
        neg_i, _ = max(diffs, key=lambda x: x[1])
        examples.append(InputExample(texts=[lines[i], lines[pos_i], lines[neg_i]]))
    loss_cls = losses.TripletLoss

else:
    # MNRL: (numeral_str, full_sentence) pairs
    for sent, age in zip(lines, ages):
        examples.append(InputExample(texts=[str(age), sent]))
    loss_cls = losses.MultipleNegativesRankingLoss


loader  = DataLoader(examples, shuffle=True, batch_size=BATCH, drop_last=True)
model   = SentenceTransformer(MODEL_ID)
loss_fn = loss_cls(model)

#from tokenizers import pre_tokenizers
#tok = model.tokenizer
#tok.backend_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.BertPreTokenizer(), pre_tokenizers.Digits(individual_digits=True)])
#print(tok.tokenize("My age is 38"))



In [ ]:
print(lines[:10])

['the 0 is the age is 65.0, the 1 is the class of worker is Employee of a private for-profit company or business, or of an individual, for wages, salary, or commissions, the 2 is the educational attainment is 1 or more years of college credit but no degree, the 3 is the marital status is Widowed, the 4 is the occupation is SAL-Sales Representatives, Wholesale And Manufacturing, the 5 is the place of birth is Alabama/AL, the 6 is the relationship is Reference person, the 7 is the usual hours worked per week past 12 months is 16.0, the 8 is the sex is Female', 'the 0 is the age is 57.0, the 1 is the class of worker is Employee of a private for-profit company or business, or of an individual, for wages, salary, or commissions, the 2 is the educational attainment is Regular high school diploma, the 3 is the marital status is Married, the 4 is the occupation is PRD-Dental And Ophthalmic Laboratory Technicians And Medical Appliance Technicians, the 5 is the place of birth is Florida/FL, the 

In [ ]:
logging.basicConfig(level=logging.INFO, handlers=[LoggingHandler()])
model.fit(
    train_objectives=[(loader, loss_fn)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(loader) * EPOCHS),
    show_progress_bar=True
)


model.save(OUT_DIR)
print(f"Saved to {OUT_DIR}")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,3.468900
1000,3.466400
1500,3.466100


Saved to bert_MNRL_ACS


In [ ]:
# 1) Zip the folder
!zip -r bert_MNRL_ACS.zip bert_MNRL_ACS

# 2) Download via browser
from google.colab import files
files.download("bert_MNRL_ACS.zip")

  adding: bert_MNRL_ACS/ (stored 0%)
  adding: bert_MNRL_ACS/model.safetensors (deflated 7%)
  adding: bert_MNRL_ACS/1_Pooling/ (stored 0%)
  adding: bert_MNRL_ACS/1_Pooling/config.json (deflated 57%)
  adding: bert_MNRL_ACS/vocab.txt (deflated 53%)
  adding: bert_MNRL_ACS/tokenizer.json (deflated 71%)
  adding: bert_MNRL_ACS/README.md (deflated 75%)
  adding: bert_MNRL_ACS/modules.json (deflated 53%)
  adding: bert_MNRL_ACS/config.json (deflated 48%)
  adding: bert_MNRL_ACS/sentence_bert_config.json (deflated 4%)
  adding: bert_MNRL_ACS/special_tokens_map.json (deflated 42%)
  adding: bert_MNRL_ACS/tokenizer_config.json (deflated 75%)
  adding: bert_MNRL_ACS/config_sentence_transformers.json (deflated 34%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>